# What If FC: bronze → silver → gold

Runs after a match has streamed into the Lakehouse (`dbo.bronze_events`, written by the
Eventstream destination `dst-bronz`). Called by the pipeline `pl-whatif-gold`.

| Layer | Table | Grain |
| --- | --- | --- |
| silver | `silver.events` | one row per (match_id, event_id): deduplicated, typed, validated |
| gold | `gold.match_summary` | one row per match: teams, score, result, completeness |
| gold | `gold.match_team_stats` | one row per match and team |
| gold | `gold.player_match_stats` | one row per match and player |
| ops | `gold.pipeline_runs` | one row per run: row counts and data-quality numbers |

Definitions of passes, completion %, shots, goals and possession proxy match the KQL
function `MatchStats()` and `analytics.metrics.match_stats()` in the repo, so the batch
numbers equal the live numbers.

Every run rebuilds silver and gold from bronze, so a rerun gives the same result
(idempotent). All clubs, players and matches are fictional.

In [ ]:
# Parameter cell: the pipeline passes p_run_id = @pipeline().RunId
p_run_id = "manual"

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

VALID_EVENT_TYPES = [
    "pass", "reception", "carry", "shot", "tackle", "interception",
    "pressure", "foul", "goal", "kickoff", "period_end",
]
POSSESSION_TYPES = ["pass", "carry", "reception", "shot"]
MATCH_ID_PATTERN = r"^match_s(\d+)_(T\d+)_(T\d+)$"


def build_silver(bronze: DataFrame) -> tuple[DataFrame, dict]:
    """Validate, type and deduplicate bronze events.

    Returns the silver DataFrame and data-quality counts.
    """
    typed = (
        bronze
        .withColumn("event_ts", F.to_timestamp("event_time"))
        .withColumn(
            "is_valid",
            F.col("event_id").isNotNull()
            & F.col("match_id").isNotNull()
            & F.col("event_type").isin(VALID_EVENT_TYPES)
            & F.col("match_clock_s").isNotNull()
            & (F.col("event_type").eqNullSafe("period_end") | F.col("team_id").isNotNull()),
        )
    )
    valid = typed.filter("is_valid")
    # Keep one row per (match_id, event_id): the same key as KQL EventsClean().
    w = Window.partitionBy("match_id", "event_id").orderBy(F.col("event_ts").asc_nulls_last())
    silver = (
        valid
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn", "is_valid", "event_time")
    )
    counts = typed.agg(
        F.count("*").alias("bronze_rows"),
        F.sum(F.when(~F.col("is_valid"), 1).otherwise(0)).alias("rejected_rows"),
    ).first()
    valid_rows = counts["bronze_rows"] - (counts["rejected_rows"] or 0)
    silver_rows = silver.count()
    dq = {
        "bronze_rows": int(counts["bronze_rows"]),
        "rejected_rows": int(counts["rejected_rows"] or 0),
        "duplicate_rows": int(valid_rows - silver_rows),
        "silver_rows": int(silver_rows),
    }
    return silver, dq


def build_team_stats(silver: DataFrame) -> DataFrame:
    """Per match and team. Same definitions as KQL MatchStats()."""
    is_pass = F.col("event_type") == "pass"
    per_team = (
        silver
        .filter(F.col("team_id").isNotNull())
        .groupBy("match_id", "team_id")
        .agg(
            F.sum(F.when(is_pass, 1).otherwise(0)).alias("passes"),
            F.sum(F.when(is_pass & (F.col("outcome") == "complete"), 1).otherwise(0))
            .alias("completed_passes"),
            F.sum(F.when(F.col("event_type") == "shot", 1).otherwise(0)).alias("shots"),
            F.sum(
                F.when(
                    (F.col("event_type") == "shot")
                    & F.col("outcome").isin("on_target", "saved"),
                    1,
                ).otherwise(0)
            ).alias("shots_on_target"),
            F.sum(F.when(F.col("event_type") == "goal", 1).otherwise(0)).alias("goals"),
            F.sum(
                F.when((F.col("event_type") == "tackle") & (F.col("outcome") == "won"), 1)
                .otherwise(0)
            ).alias("tackles_won"),
            F.sum(F.when(F.col("event_type") == "interception", 1).otherwise(0))
            .alias("interceptions"),
            F.round(F.avg(F.when(is_pass, F.col("pressure_level"))), 3)
            .alias("avg_pressure_on_passes"),
            F.sum(F.when(F.col("event_type").isin(POSSESSION_TYPES), 1).otherwise(0))
            .alias("possession_events"),
        )
    )
    match_poss = Window.partitionBy("match_id")
    total = F.sum("possession_events").over(match_poss)
    return (
        per_team
        .withColumn(
            "completion_pct",
            F.when(F.col("passes") > 0,
                   F.round(100.0 * F.col("completed_passes") / F.col("passes"), 1))
            .otherwise(F.lit(0.0)),
        )
        .withColumn(
            "possession_proxy_pct",
            F.when(total > 0, F.round(100.0 * F.col("possession_events") / total, 1))
            .otherwise(F.lit(0.0)),
        )
        .drop("possession_events")
        .select(
            "match_id", "team_id", "passes", "completed_passes", "completion_pct",
            "shots", "shots_on_target", "goals", "possession_proxy_pct",
            "tackles_won", "interceptions", "avg_pressure_on_passes",
        )
    )


def build_match_summary(silver: DataFrame, team_stats: DataFrame) -> DataFrame:
    """One row per match: teams parsed from match_id, score, result, completeness."""
    base = (
        silver
        .groupBy("match_id")
        .agg(
            F.first("seed", ignorenulls=True).alias("seed"),
            F.count("*").alias("total_events"),
            F.max("match_clock_s").alias("last_clock_s"),
            F.min("event_ts").alias("first_event_ts"),
            F.max("event_ts").alias("last_event_ts"),
            F.max(
                F.when((F.col("event_type") == "period_end") & (F.col("period") == 2), 1)
                .otherwise(0)
            ).alias("_ended"),
        )
        .withColumn("home_team_id", F.regexp_extract("match_id", MATCH_ID_PATTERN, 2))
        .withColumn("away_team_id", F.regexp_extract("match_id", MATCH_ID_PATTERN, 3))
        .withColumn("is_complete", F.col("_ended") == 1)
        .drop("_ended")
    )
    goals = team_stats.select("match_id", "team_id", "goals")
    home = goals.withColumnRenamed("team_id", "home_team_id").withColumnRenamed("goals", "home_goals")
    away = goals.withColumnRenamed("team_id", "away_team_id").withColumnRenamed("goals", "away_goals")
    return (
        base
        .join(home, ["match_id", "home_team_id"], "left")
        .join(away, ["match_id", "away_team_id"], "left")
        .fillna(0, ["home_goals", "away_goals"])
        .withColumn(
            "result",
            F.when(F.col("home_goals") > F.col("away_goals"), "home_win")
            .when(F.col("home_goals") < F.col("away_goals"), "away_win")
            .otherwise("draw"),
        )
        .withColumn("duration_min", F.round(F.col("last_clock_s") / 60.0, 1))
        .select(
            "match_id", "seed", "home_team_id", "away_team_id", "home_goals", "away_goals",
            "result", "total_events", "duration_min", "is_complete",
            "first_event_ts", "last_event_ts",
        )
    )


def build_player_stats(silver: DataFrame) -> DataFrame:
    """One row per match and player."""
    is_pass = F.col("event_type") == "pass"
    stats = (
        silver
        .filter(F.col("player_id").isNotNull())
        .groupBy("match_id", "team_id", "player_id")
        .agg(
            F.sum(F.when(is_pass, 1).otherwise(0)).alias("passes"),
            F.sum(F.when(is_pass & (F.col("outcome") == "complete"), 1).otherwise(0))
            .alias("completed_passes"),
            F.sum(F.when(F.col("event_type") == "reception", 1).otherwise(0)).alias("receptions"),
            F.sum(F.when(F.col("event_type") == "carry", 1).otherwise(0)).alias("carries"),
            F.sum(F.when(F.col("event_type") == "shot", 1).otherwise(0)).alias("shots"),
            F.sum(F.when(F.col("event_type") == "goal", 1).otherwise(0)).alias("goals"),
            F.sum(
                F.when((F.col("event_type") == "tackle") & (F.col("outcome") == "won"), 1)
                .otherwise(0)
            ).alias("tackles_won"),
            F.sum(F.when(F.col("event_type") == "interception", 1).otherwise(0))
            .alias("interceptions"),
        )
    )
    return stats.withColumn(
        "completion_pct",
        F.when(F.col("passes") > 0,
               F.round(100.0 * F.col("completed_passes") / F.col("passes"), 1))
        .otherwise(F.lit(0.0)),
    )

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

bronze = spark.read.table("dbo.bronze_events")
silver, dq = build_silver(bronze)
silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.events")
print(dq)

In [ ]:
silver = spark.read.table("silver.events")
team_stats = build_team_stats(silver)
summary = build_match_summary(silver, team_stats)
players = build_player_stats(silver)

for name, df in [
    ("gold.match_team_stats", team_stats),
    ("gold.match_summary", summary),
    ("gold.player_match_stats", players),
]:
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(name, df.count(), "rows")

In [ ]:
from datetime import datetime, timezone

run_row = spark.createDataFrame(
    [(
        p_run_id,
        datetime.now(timezone.utc),
        dq["bronze_rows"],
        dq["rejected_rows"],
        dq["duplicate_rows"],
        dq["silver_rows"],
        summary.count(),
    )],
    "run_id string, processed_at timestamp, bronze_rows long, rejected_rows long, "
    "duplicate_rows long, silver_rows long, matches long",
)
run_row.write.mode("append").saveAsTable("gold.pipeline_runs")
display(spark.read.table("gold.pipeline_runs").orderBy(F.desc("processed_at")).limit(5))

In [ ]:
# Check: compare with KQL in the Eventhouse, e.g. MatchStats("match_s7_T01_T02").
# passes, completed_passes, completion_pct, shots, goals and possession_proxy_pct must match.
display(spark.read.table("gold.match_summary").orderBy("match_id"))
display(spark.read.table("gold.match_team_stats").orderBy("match_id", "team_id"))